# Dry Gas–Gas Bare Tube Heat Exchanger Test

Functional test case for a gas–gas heat exchanger on smooth bare tubes.

This notebook is intentionally a **gas-phase / sensible-only** test case:

- no PsychroLib,
- no `MoistAirState`,
- no relative humidity,
- no dew point,
- no condensation,
- no wet-surface heat transfer.

Model selection:

- `dry_air` uses `dry_air_props()` / `DryAirPropertyProvider`,
- `gas_mixture` uses `GasMixtureSpec` / `GasMixturePropertyProvider`.

The outside gas-mixture example may include `H2O` as an explicit gas-phase
component. This still means gas-phase sensible calculation only. Water removal,
latent heat, and condensation are not modelled.

Requested result outputs include inlet and outlet actual volume flows.

In [ ]:
from pathlib import Path
import sys
import math

import pandas as pd

# Make the notebook usable both from repository root and from core/tests.
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]

for candidate in candidate_roots:
    if (candidate / "core").is_dir():
        workspace_root = candidate
        break
else:
    raise RuntimeError("Could not find repository root containing the 'core' directory.")

if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

print("Workspace root:", workspace_root)

In [ ]:
from core.common.flow import volume_flow_from_mass_flow, volume_flow_m3_h_from_mass_flow

from core.properties import (
    GasMixturePropertyProvider,
    GasMixtureSpec,
    dry_air_props,
)

from core.properties.adapters import (
    to_internal_fluid_props,
    to_internal_pressure_drop_fluid_props,
    to_outside_fluid_props,
)

print("Imports completed.")

## Input Data

In [ ]:
# ---------------------------------------------------------------------------
# Global conditions
# ---------------------------------------------------------------------------

p_atm = 101325.0
T0_C = 273.15

# ---------------------------------------------------------------------------
# Cold dry gas, inside tubes
# ---------------------------------------------------------------------------

inside_gas_mode = "dry_air"      # options: "dry_air", "gas_mixture"

inside_T_in_C = 30.0
inside_T_in = inside_T_in_C + T0_C
inside_p_in = p_atm
inside_p_out = p_atm             # pressure drop is only estimated separately in this notebook

inside_m_dot_kg_h = 18_220.0
inside_m_dot = inside_m_dot_kg_h / 3600.0

# Prepared user-defined dry gas mixture.
# Edit this composition to replace dry air inside tubes.
# Used only when inside_gas_mode = "gas_mixture".
inside_gas_mixture_components = {
    "N2": 0.79,
    "O2": 0.21,
}

# ---------------------------------------------------------------------------
# Hot dry / gas-phase mixture, outside tubes
# ---------------------------------------------------------------------------

outside_gas_mode = "gas_mixture"     # options: "dry_air", "gas_mixture"

outside_T_in_C = 400.0
outside_T_in = outside_T_in_C + T0_C
outside_p_in = p_atm
outside_p_out = p_atm            # pressure drop is not solved here

outside_m_dot_kg_h = 28_380.0
outside_m_dot = outside_m_dot_kg_h / 3600.0

# Prepared user-defined dry hot gas / gas-phase mixture.
# Edit this composition to replace dry air outside tubes.
# Used only when outside_gas_mode = "gas_mixture".
#
# Example below: dry-air basis 79/21 with 120 g H2O/kg dry air,
# represented as wet mole fractions:
# N2  = 0.6627
# O2  = 0.1761
# H2O = 0.1612
outside_gas_mixture_components = {
    "N2": 0.6627,
    "O2": 0.1761,
    "H2O": 0.1612,
}

# ---------------------------------------------------------------------------
# Bare tube bundle geometry
# ---------------------------------------------------------------------------

tube_Do = 25.0e-3
tube_wall = 1.5e-3
tube_Di = tube_Do - 2.0 * tube_wall

tube_length = 2800.0e-3
tubes_per_row = 56
n_rows = 36
tube_passes = 2

pitch_transverse = 35.0e-3
pitch_longitudinal = 35.0e-3
arrangement = "inline"

n_tubes_total = tubes_per_row * n_rows

if n_tubes_total % tube_passes != 0:
    raise ValueError("Total tube count must be divisible by number of tube passes for this test.")

n_parallel_tubes_per_pass = n_tubes_total // tube_passes

# Carbon steel / black steel, approximate engineering value.
tube_material = "carbon steel"
tube_k = 50.0  # W/(m*K)

# Stainless steel test alternative:
# tube_material = "stainless steel"
# tube_k = 16.0  # W/(m*K)

inputs = {
    "inside_gas_mode": inside_gas_mode,
    "inside_T_in_C": inside_T_in_C,
    "inside_m_dot_kg_h": inside_m_dot_kg_h,
    "outside_gas_mode": outside_gas_mode,
    "outside_T_in_C": outside_T_in_C,
    "outside_m_dot_kg_h": outside_m_dot_kg_h,
    "tube_Do_mm": tube_Do * 1000.0,
    "tube_wall_mm": tube_wall * 1000.0,
    "tube_Di_mm": tube_Di * 1000.0,
    "tube_length_mm": tube_length * 1000.0,
    "tubes_per_row": tubes_per_row,
    "n_rows": n_rows,
    "n_tubes_total": n_tubes_total,
    "tube_passes": tube_passes,
    "n_parallel_tubes_per_pass": n_parallel_tubes_per_pass,
    "pitch_transverse_mm": pitch_transverse * 1000.0,
    "pitch_longitudinal_mm": pitch_longitudinal * 1000.0,
    "arrangement": arrangement,
    "tube_material": tube_material,
    "tube_k_W_mK": tube_k,
}

pd.Series(inputs, name="value").to_frame()

## Medium Selection

Only `gas_mixture` modes use user-defined compositions.

`dry_air` mode is handled by the dedicated dry-air provider and does not require
a user-defined `N2/O2` composition.

In [ ]:
def make_inside_gas_spec() -> GasMixtureSpec:
    if inside_gas_mode != "gas_mixture":
        raise ValueError("Inside GasMixtureSpec is only used for inside_gas_mode='gas_mixture'.")

    return GasMixtureSpec(
        components=inside_gas_mixture_components,
        basis="mole",
        backend="HEOS",
        imposed_phase="gas",
    )


def make_outside_gas_spec() -> GasMixtureSpec:
    if outside_gas_mode != "gas_mixture":
        raise ValueError("Outside GasMixtureSpec is only used for outside_gas_mode='gas_mixture'.")

    return GasMixtureSpec(
        components=outside_gas_mixture_components,
        basis="mole",
        backend="HEOS",
        imposed_phase="gas",
    )


def inside_props(T: float, p: float):
    if inside_gas_mode == "dry_air":
        props = dry_air_props(T=T, p=p)
        return props, "dry_air_props", "DryAirPropertyProvider"

    if inside_gas_mode == "gas_mixture":
        spec = make_inside_gas_spec()
        props = GasMixturePropertyProvider(spec).at(T=T, p=p)
        return props, "GasMixturePropertyProvider", spec.to_mole_fractions()

    raise ValueError(f"Unsupported inside_gas_mode: {inside_gas_mode!r}")


def outside_props(T: float, p: float):
    if outside_gas_mode == "dry_air":
        props = dry_air_props(T=T, p=p)
        return props, "dry_air_props", "DryAirPropertyProvider"

    if outside_gas_mode == "gas_mixture":
        spec = make_outside_gas_spec()
        props = GasMixturePropertyProvider(spec).at(T=T, p=p)
        return props, "GasMixturePropertyProvider", spec.to_mole_fractions()

    raise ValueError(f"Unsupported outside_gas_mode: {outside_gas_mode!r}")

## Property Evaluation at Inlet

In [ ]:
inside_props_in, inside_property_model, inside_composition_or_source = inside_props(
    T=inside_T_in,
    p=inside_p_in,
)

outside_props_in, outside_property_model, outside_composition_or_source = outside_props(
    T=outside_T_in,
    p=outside_p_in,
)

# Adapter objects for core solver compatibility.
inside_internal_props = to_internal_fluid_props(inside_props_in)
inside_internal_dp_props = to_internal_pressure_drop_fluid_props(inside_props_in)
outside_crossflow_props = to_outside_fluid_props(outside_props_in)

inside_Vdot_in = volume_flow_from_mass_flow(
    m_dot=inside_m_dot,
    rho=inside_props_in.rho,
)

outside_Vdot_in = volume_flow_from_mass_flow(
    m_dot=outside_m_dot,
    rho=outside_props_in.rho,
)

property_rows = [
    {
        "side": "inside tubes",
        "mode": inside_gas_mode,
        "property_model": inside_property_model,
        "composition_or_source": inside_composition_or_source,
        "T_in_C": inside_T_in_C,
        "p_in_bar": inside_p_in / 1e5,
        "m_dot_kg_h": inside_m_dot * 3600.0,
        "Vdot_in_m3_h": inside_Vdot_in * 3600.0,
        "rho_in_kg_m3": inside_props_in.rho,
        "mu_in_Pa_s": inside_props_in.mu,
        "k_in_W_mK": inside_props_in.k,
        "cp_in_J_kgK": inside_props_in.cp,
        "Pr_in": inside_props_in.mu * inside_props_in.cp / inside_props_in.k,
    },
    {
        "side": "outside crossflow",
        "mode": outside_gas_mode,
        "property_model": outside_property_model,
        "composition_or_source": outside_composition_or_source,
        "T_in_C": outside_T_in_C,
        "p_in_bar": outside_p_in / 1e5,
        "m_dot_kg_h": outside_m_dot * 3600.0,
        "Vdot_in_m3_h": outside_Vdot_in * 3600.0,
        "rho_in_kg_m3": outside_props_in.rho,
        "mu_in_Pa_s": outside_props_in.mu,
        "k_in_W_mK": outside_props_in.k,
        "cp_in_J_kgK": outside_props_in.cp,
        "Pr_in": outside_props_in.mu * outside_props_in.cp / outside_props_in.k,
    },
]

property_df = pd.DataFrame(property_rows)

assert inside_m_dot > 0.0
assert outside_m_dot > 0.0
assert inside_props_in.rho > 0.0
assert outside_props_in.rho > 0.0
assert inside_Vdot_in > 0.0
assert outside_Vdot_in > 0.0

property_df

## Temporary 0D Heat-Exchanger Sanity Check

This cell provides outlet temperatures only so that outlet volume flows can be
checked in this notebook.

It is intentionally marked as temporary. The final validated rating path should
come from the KalKalori bare-tube heat-transfer and pressure-drop API, not from
notebook-local correlations.

In [ ]:
def friction_factor_smooth_pipe(Re: float) -> float:
    if Re <= 0.0:
        raise ValueError("Re must be positive.")
    if Re < 2300.0:
        return 64.0 / Re
    return 0.3164 * Re ** (-0.25)


def nusselt_internal_smooth_tube(Re: float, Pr: float) -> float:
    if Re <= 0.0 or Pr <= 0.0:
        raise ValueError("Re and Pr must be positive.")
    if Re < 2300.0:
        return 3.66
    return 0.023 * Re ** 0.8 * Pr ** 0.4


def zukauskas_inline_tube_bank_Nu(Re: float, Pr: float, n_rows: int) -> float:
    if Re <= 0.0 or Pr <= 0.0:
        raise ValueError("Re and Pr must be positive.")

    if Re < 100.0:
        C, m = 0.9, 0.4
    elif Re < 1000.0:
        C, m = 0.52, 0.5
    elif Re < 2.0e5:
        C, m = 0.27, 0.63
    else:
        C, m = 0.021, 0.84

    Nu = C * Re ** m * Pr ** 0.36

    if n_rows < 20:
        C2 = {
            1: 0.7,
            2: 0.8,
            3: 0.86,
            4: 0.9,
            5: 0.92,
            6: 0.94,
            7: 0.96,
            8: 0.97,
            9: 0.98,
            10: 0.99,
        }.get(n_rows, 1.0)
        Nu *= C2

    return Nu


def epsilon_crossflow_both_unmixed(NTU: float, Cr: float) -> float:
    if NTU <= 0.0:
        return 0.0
    if Cr <= 0.0:
        return 1.0 - math.exp(-NTU)
    return 1.0 - math.exp((math.exp(-Cr * NTU ** 0.78) - 1.0) / (Cr * NTU ** -0.22))

In [ ]:
# Tube-side geometry.
area_flow_one_tube = math.pi * tube_Di ** 2 / 4.0
inside_flow_area_total = n_parallel_tubes_per_pass * area_flow_one_tube
inside_velocity = inside_m_dot / (inside_props_in.rho * inside_flow_area_total)

inside_Re = inside_props_in.rho * inside_velocity * tube_Di / inside_props_in.mu
inside_Pr = inside_props_in.mu * inside_props_in.cp / inside_props_in.k
inside_Nu = nusselt_internal_smooth_tube(inside_Re, inside_Pr)
inside_alfa = inside_Nu * inside_props_in.k / tube_Di

inside_f = friction_factor_smooth_pipe(inside_Re)
inside_total_tube_length_per_flow_path = tube_length * tube_passes
inside_dp_friction = (
    inside_f
    * inside_total_tube_length_per_flow_path
    / tube_Di
    * 0.5
    * inside_props_in.rho
    * inside_velocity ** 2
)

# Outside tube-bank geometry.
outside_frontal_area = tube_length * tubes_per_row * pitch_transverse
outside_min_free_area = tube_length * tubes_per_row * (pitch_transverse - tube_Do)
outside_face_velocity = outside_m_dot / (outside_props_in.rho * outside_frontal_area)
outside_vmax = outside_m_dot / (outside_props_in.rho * outside_min_free_area)

outside_Re = outside_props_in.rho * outside_vmax * tube_Do / outside_props_in.mu
outside_Pr = outside_props_in.mu * outside_props_in.cp / outside_props_in.k
outside_Nu = zukauskas_inline_tube_bank_Nu(outside_Re, outside_Pr, n_rows)
outside_alfa = outside_Nu * outside_props_in.k / tube_Do

# Area and wall resistance.
A_o = n_tubes_total * math.pi * tube_Do * tube_length
wall_R_o_basis = tube_Do * math.log(tube_Do / tube_Di) / (2.0 * tube_k)

Uo = 1.0 / (
    1.0 / outside_alfa
    + wall_R_o_basis
    + tube_Do / (tube_Di * inside_alfa)
)

# Capacity rates.
C_inside = inside_m_dot * inside_props_in.cp
C_outside = outside_m_dot * outside_props_in.cp

C_min = min(C_inside, C_outside)
C_max = max(C_inside, C_outside)
Cr = C_min / C_max

UA = Uo * A_o
NTU = UA / C_min
epsilon = epsilon_crossflow_both_unmixed(NTU, Cr)

# Hot gas outside, cold gas inside.
q_max = C_min * (outside_T_in - inside_T_in)
q = epsilon * q_max

inside_T_out = inside_T_in + q / C_inside
outside_T_out = outside_T_in - q / C_outside

assert inside_T_out > inside_T_in
assert outside_T_out < outside_T_in

summary = {
    "q_kW": q / 1000.0,
    "inside_T_out_C": inside_T_out - T0_C,
    "outside_T_out_C": outside_T_out - T0_C,
    "inside_Re": inside_Re,
    "outside_Re": outside_Re,
    "inside_alfa_W_m2K": inside_alfa,
    "outside_alfa_W_m2K": outside_alfa,
    "inside_dp_friction_Pa": inside_dp_friction,
    "A_o_m2": A_o,
    "Uo_W_m2K": Uo,
    "UA_W_K": UA,
    "Cr": Cr,
    "NTU": NTU,
    "epsilon": epsilon,
    "tube_material": tube_material,
    "tube_k_W_mK": tube_k,
}

assert q > 0.0
assert Uo > 0.0
assert 0.0 <= epsilon <= 1.0

pd.Series(summary, name="value").to_frame()

## Inlet and Outlet Volume Flows

Outlet properties are evaluated at calculated outlet temperatures with the same
composition and pressure assumptions as at inlet.

In [ ]:
inside_props_out, _, _ = inside_props(
    T=inside_T_out,
    p=inside_p_out,
)

outside_props_out, _, _ = outside_props(
    T=outside_T_out,
    p=outside_p_out,
)

inside_Vdot_out = volume_flow_from_mass_flow(
    m_dot=inside_m_dot,
    rho=inside_props_out.rho,
)

outside_Vdot_out = volume_flow_from_mass_flow(
    m_dot=outside_m_dot,
    rho=outside_props_out.rho,
)

results_rows = [
    {
        "side": "inside tubes",
        "mode": inside_gas_mode,
        "property_model": inside_property_model,
        "m_dot_kg_h": inside_m_dot * 3600.0,
        "T_in_C": inside_T_in - T0_C,
        "T_out_C": inside_T_out - T0_C,
        "rho_in_kg_m3": inside_props_in.rho,
        "rho_out_kg_m3": inside_props_out.rho,
        "Vdot_in_m3_h": inside_Vdot_in * 3600.0,
        "Vdot_out_m3_h": inside_Vdot_out * 3600.0,
        "Vdot_in_m3_s": inside_Vdot_in,
        "Vdot_out_m3_s": inside_Vdot_out,
    },
    {
        "side": "outside crossflow",
        "mode": outside_gas_mode,
        "property_model": outside_property_model,
        "m_dot_kg_h": outside_m_dot * 3600.0,
        "T_in_C": outside_T_in - T0_C,
        "T_out_C": outside_T_out - T0_C,
        "rho_in_kg_m3": outside_props_in.rho,
        "rho_out_kg_m3": outside_props_out.rho,
        "Vdot_in_m3_h": outside_Vdot_in * 3600.0,
        "Vdot_out_m3_h": outside_Vdot_out * 3600.0,
        "Vdot_in_m3_s": outside_Vdot_in,
        "Vdot_out_m3_s": outside_Vdot_out,
    },
]

results_df = pd.DataFrame(results_rows)

assert (results_df["Vdot_in_m3_h"] > 0.0).all()
assert (results_df["Vdot_out_m3_h"] > 0.0).all()

results_df